In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# Getting the info and describe of the training data
train_path = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
test_path = '/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"The Training Data shape is -> {train_df.shape}")
print(f"The Testing Data shape is -> {test_df.shape}")

In [ ]:
# Now getting the training data info and the describe on the training data
print(f"The Training Data looks like this -> \n{train_df.head()}")
print(f"\nThe Description about the training data is -> \n{train_df.describe()}")
print(f"\nThe value counts of the training data is -> \n{train_df['answer'].value_counts()}")

In [ ]:
import matplotlib.pyplot as plt

# 1. Get the value counts of the answers
answer_counts = train_df['answer'].value_counts()

# 2. Set up the data, labels, and visual styling
labels = answer_counts.index
sizes = answer_counts.values
colors = ['#4f46e5', '#06b6d4', '#10b981', '#f59e0b', '#ef4444']  # Modern, clean color palette
explode = (0.05, 0, 0, 0, 0)  # Slightly separate the top answer (B) for visual emphasis

# 3. Create the pie chart
plt.figure(figsize=(8, 6))
plt.pie(
    sizes, 
    explode=explode, 
    labels=labels, 
    colors=colors, 
    autopct='%1.1f%%',  # Shows the percentages automatically
    startangle=140,     # Rotates the chart for a nicer layout
    textprops={'fontsize': 12, 'weight': 'bold'},
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}  # Adds clean borders between slices
)

# 4. Add titles and adjustments
plt.title("Distribution of Correct Answers in Training Data", fontsize=14, weight='bold', pad=20)
plt.axis('equal')  # Ensures the pie chart is drawn as a perfect circle

# 5. Display the plot
plt.show()

Milestone 2

In [ ]:
# # Install required Hugging Face libraries if missing
# !pip install -q datasets sentence-transformers

# import os
# import re
# import numpy as np
# import pandas as pd
# import torch
# from datasets import load_dataset
# from transformers import AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM, pipeline
# from sentence_transformers import SentenceTransformer, util
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics.pairwise import cosine_similarity

# # Determine the dataset path on Kaggle
# DATA_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
# if not os.path.exists(DATA_PATH):
#     DATA_PATH = "train.csv"  # Local fallback

# print(f"Using dataset path: {DATA_PATH}\n")

# # --- Q1: HF datasets length of combined_text ---
# print("--- Task 1: HF datasets length ---")
# dataset = load_dataset("csv", data_files=DATA_PATH, split="train")

# def concatenate_prompt_A(example):
#     example["combined_text"] = f"{example['prompt']} {example['A']}"
#     return example

# dataset = dataset.map(concatenate_prompt_A)
# q1_length = len(dataset[51]["combined_text"])
# print(f"-> Answer: {q1_length}\n")

# # --- Q2 & Q3: Tokenizer configs ---
# print("--- Task 2 & 3: Tokenizer parameters ---")
# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# print(f"-> Answer Q2 (Vocab size): {tokenizer.vocab_size}")
# print(f"-> Answer Q3 ([SEP] ID): {tokenizer.convert_tokens_to_ids('[SEP]')}\n")

# # --- Q4: Shape of tokenized prompt ---
# print("--- Task 4: Input IDs Shape ---")
# prompts = [str(x) if x is not None else "" for x in dataset["prompt"]]
# tokenized_prompts = tokenizer(
#     prompts,
#     padding="max_length",
#     truncation=True,
#     max_length=128,
#     return_tensors="pt"
# )
# print(f"-> Answer: {list(tokenized_prompts['input_ids'].shape)}\n")

# # --- Q5: Attention head dimensionality ---
# print("--- Task 5: Attention head dim ---")
# # 768 hidden dimensions / 12 heads
# print(f"-> Answer: {768 // 12}\n")

# # --- Q6 & Q7: BERT Output shape & CLS vector ---
# print("--- Task 6 & 7: BERT CLS embeddings ---")
# row_0_prompt = dataset[0]["prompt"]
# inputs_row_0 = tokenizer(row_0_prompt, return_tensors="pt")
# model = AutoModel.from_pretrained("bert-base-uncased")
# model.eval()
# with torch.no_grad():
#     outputs_row_0 = model(**inputs_row_0)
    
# last_hidden = outputs_row_0.last_hidden_state
# cls_vector = last_hidden[0, 0].numpy()
# print(f"-> Answer Q6 (Shape): {list(last_hidden.shape)}")
# print(f"-> Answer Q7 (Sum of first 5 floats): {round(float(np.sum(cls_vector[:5])), 4)}\n")

# # --- Q8: Attention weights ---
# print("--- Task 8: Attention weights ---")
# model_att = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)
# model_att.eval()

# sentence = "Light-ion fusion is a technique."
# inputs_att = tokenizer(sentence, return_tensors="pt")
# input_ids = inputs_att["input_ids"][0].tolist()
# tokens = tokenizer.convert_ids_to_tokens(input_ids)

# fusion_idx = tokens.index("fusion")
# with torch.no_grad():
#     outputs_att = model_att(**inputs_att)
    
# # Last layer (-1), First Head (0), CLS token (0) to fusion token
# cls_to_fusion = outputs_att.attentions[-1][0, 0, 0, fusion_idx].item()
# print(f"-> Answer: {round(cls_to_fusion, 4)}\n")

# # --- Q9: Cosine similarity using Sentence Transformers ---
# print("--- Task 9: Sentence Transformers Cosine Similarity ---")
# st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
# row_0_b = dataset[0]["B"]

# p_emb = st_model.encode(row_0_prompt, convert_to_tensor=True)
# b_emb = st_model.encode(row_0_b, convert_to_tensor=True)
# cos_sim = util.cos_sim(p_emb, b_emb).item()
# print(f"-> Answer: {round(cos_sim, 4)}\n")

# # --- Q10: MAP@3 comparison pipeline ---
# print("--- Task 10: TF-IDF vs SentenceTransformer MAP@3 ---")
# train_df = pd.read_csv(DATA_PATH).fillna("")

# # Preprocessing helper
# def clean_text_simple(text):
#     text = text.lower()
#     text = re.sub(r"[^\w\s]", "", text)
#     text = re.sub(r"\s+", " ", text).strip()
#     return text

# # Pipeline 1: TF-IDF
# tfidf_preds = []
# for idx, row in train_df.iterrows():
#     p_clean = clean_text_simple(row['prompt'])
#     choices = [clean_text_simple(row[c]) for c in ['A', 'B', 'C', 'D', 'E']]
#     vectorizer = TfidfVectorizer()
#     try:
#         vectors = vectorizer.fit_transform([p_clean] + choices).toarray()
#         sims = cosine_similarity(vectors[0:1], vectors[1:])[0]
#     except:
#         sims = np.zeros(5)
#     sorted_indices = np.argsort(sims)[::-1]
#     tfidf_preds.append([['A', 'B', 'C', 'D', 'E'][i] for i in sorted_indices[:3]])

# # Pipeline 2: MiniLM
# minilm_preds = []
# prompt_embs = st_model.encode(train_df['prompt'].tolist(), show_progress_bar=False)
# for idx, row in train_df.iterrows():
#     p_emb = prompt_embs[idx]
#     choice_embs = st_model.encode([row[c] for c in ['A', 'B', 'C', 'D', 'E']])
#     sims = [util.cos_sim(p_emb, c_emb).item() for c_emb in choice_embs]
#     sorted_indices = np.argsort(sims)[::-1]
#     minilm_preds.append([['A', 'B', 'C', 'D', 'E'][i] for i in sorted_indices[:3]])

# # Evaluate MAP@3
# targets = train_df['answer'].tolist()
# ap_scores = []
# for p, t in zip(minilm_preds, targets):
#     score = 0.0
#     for r, choice in enumerate(p[:3]):
#         if choice == t:
#             score = 1.0 / (r + 1)
#             break
#     ap_scores.append(score)
# minilm_map3 = np.mean(ap_scores)

# # Count overlaps
# count_overlap = 0
# for idx, t in enumerate(targets):
#     in_tfidf = t in tfidf_preds[idx]
#     in_minilm = t in minilm_preds[idx]
#     if (not in_tfidf) and in_minilm:
#         count_overlap += 1

# print(f"-> Answer Q10a (MiniLM MAP@3): {round(minilm_map3, 4)}")
# print(f"-> Answer Q10b (Not in TF-IDF but in MiniLM): {count_overlap}\n")

# # --- Q11 & Q12: Zero-shot classification ---
# print("--- Task 11 & 12: Zero-Shot Classification ---")
# classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
# row_1_prompt = dataset[1]["prompt"]
# row_1_labels = [dataset[1]["A"], dataset[1]["B"], dataset[1]["C"]]

# # Single-label (Softmax)
# res_softmax = classifier(row_1_prompt, candidate_labels=row_1_labels, multi_label=False)
# print(f"-> Answer Q11 (Top score Softmax): {round(res_softmax['scores'][0], 4)}")

# # Multi-label (Independent Sigmoids)
# res_sigmoid = classifier(row_1_prompt, candidate_labels=row_1_labels, multi_label=True)
# sum_softmax = sum(res_softmax["scores"])
# sum_sigmoid = sum(res_sigmoid["scores"])
# print(f"-> Answer Q12 (Absolute difference): {round(abs(sum_softmax - sum_sigmoid), 4)}\n")

# # --- Q13: Seq2Seq Generative QA ---
# print("--- Task 13: Seq2Seq Text2Text Generation ---")
# model_t5 = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")
# tokenizer_t5 = AutoTokenizer.from_pretrained("google/flan-t5-small")

# row = dataset[0]
# input_str = f"Question: {row['prompt']}. Is the correct answer A: {row['A']} or B: {row['B']}? Answer with just the letter A or B."
# inputs = tokenizer_t5(input_str, return_tensors="pt")
# outputs = model_t5.generate(**inputs, max_new_tokens=5)
# t5_out = tokenizer_t5.decode(outputs[0], skip_special_tokens=True).strip()
# print(f"-> Answer: {t5_out}\n")

In [ ]:
# Making our submission as a dummy (Baseline)
# Creating the top 3 most occuring options (the modal values) then converting them to a list
# After we convert to a list we essentially convert to a string value thats necessary for the submission formatimport pandas as pd

# Reading the Competition Data
train_df = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)

test_df = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
)

print("Competition Data Loaded Successfully !!")
t_3 = train_df['answer'].value_counts().index[:3].tolist()
pred = ' '.join(t_3)
print(f"The Baseline Prediction is -> {pred}")

# And now we make a DataFrame for the dummy submission
dummy_df = pd.DataFrame({
    'ID' : test_df['id'],
    'Prediction' : pred
})

# Now we convert the DataFrame to a CSV (comma separated values) for the submission format
dummy_df.to_csv(
   'submission.csv',
    index = False
)
print("Baseline Sumbission CSV is here !!")

In [ ]:
# import os
# import re
# import numpy as np
# import pandas as pd
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import Dataset, DataLoader
# from sklearn.model_selection import train_test_split

# # ----------------------------------------------------
# # 1. Configuration & Paths
# # ----------------------------------------------------
# TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
# TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
# OUTPUT_PATH = "submission.csv"

# MAX_LEN = 128
# BATCH_SIZE = 32
# EPOCHS = 5
# LEARNING_RATE = 0.001

# # Select GPU if available
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")

# # Special Tokens
# PAD_TOKEN = "<PAD>"
# UNK_TOKEN = "<UNK>"
# SEP_TOKEN = "<SEP>"

# # ----------------------------------------------------
# # 2. Text Preprocessing & Vocabulary Build
# # ----------------------------------------------------
# def clean_text(text):
#     if not isinstance(text, str):
#         return ""
#     text = text.lower()
#     text = re.sub(r"[^\w\s]", "", text)  # remove special chars/punctuation
#     text = re.sub(r"\s+", " ", text).strip()  # normalize whitespaces
#     return text

# class MCQVocabulary:
#     def __init__(self, max_vocab_size=15000):
#         self.max_vocab_size = max_vocab_size
#         self.word2idx = {PAD_TOKEN: 0, UNK_TOKEN: 1, SEP_TOKEN: 2}
#         self.idx2word = {0: PAD_TOKEN, 1: UNK_TOKEN, 2: SEP_TOKEN}
        
#     def fit(self, texts):
#         word_counts = {}
#         for text in texts:
#             cleaned = clean_text(text)
#             for word in cleaned.split():
#                 word_counts[word] = word_counts.get(word, 0) + 1
                
#         # Sort words by frequency
#         sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
#         for word, count in sorted_words[:self.max_vocab_size]:
#             if word not in self.word2idx:
#                 idx = len(self.word2idx)
#                 self.word2idx[word] = idx
#                 self.idx2word[idx] = word
                
#     def encode(self, text, max_len=128):
#         cleaned = clean_text(text)
#         tokens = cleaned.split()
#         encoded = [self.word2idx.get(w, self.word2idx[UNK_TOKEN]) for w in tokens]
        
#         # Truncate
#         if len(encoded) > max_len:
#             encoded = encoded[:max_len]
#         # Pad
#         else:
#             encoded = encoded + [self.word2idx[PAD_TOKEN]] * (max_len - len(encoded))
#         return encoded

# # ----------------------------------------------------
# # 3. PyTorch Dataset & DataLoader Setup
# # ----------------------------------------------------
# class ScratchMCQDataset(Dataset):
#     def __init__(self, df, vocab, max_len=128, is_train=True):
#         self.df = df.reset_index(drop=True)
#         self.vocab = vocab
#         self.max_len = max_len
#         self.is_train = is_train
#         self.label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
        
#     def __len__(self):
#         return len(self.df)
        
#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]
#         prompt = str(row['prompt'])
        
#         # Build prompt + SEP + option for A, B, C, D, E
#         input_ids = []
#         for choice in ['A', 'B', 'C', 'D', 'E']:
#             choice_text = str(row[choice])
#             combined_text = f"{prompt} {SEP_TOKEN} {choice_text}"
#             encoded = self.vocab.encode(combined_text, max_len=self.max_len)
#             input_ids.append(encoded)
            
#         input_ids = torch.tensor(input_ids, dtype=torch.long)  # [5, max_len]
        
#         if self.is_train and 'answer' in row:
#             label = torch.tensor(self.label_map[row['answer']], dtype=torch.long)
#             return input_ids, label
#         return input_ids

# # ----------------------------------------------------
# # 4. Neural Network Architecture from Scratch
# # ----------------------------------------------------
# class SelfAttentionPooling(nn.Module):
#     def __init__(self, hidden_dim):
#         super().__init__()
#         self.attention = nn.Sequential(
#             nn.Linear(hidden_dim, hidden_dim // 2),
#             nn.Tanh(),
#             nn.Linear(hidden_dim // 2, 1)
#         )
        
#     def forward(self, rnn_outputs):
#         # Input shape: [batch_size, seq_len, hidden_dim]
#         weights = self.attention(rnn_outputs)  # [batch_size, seq_len, 1]
#         weights = torch.softmax(weights, dim=1)
#         pooled = torch.sum(rnn_outputs * weights, dim=1)  # [batch_size, hidden_dim]
#         return pooled

# class BiGRUAttentionMCQModel(nn.Module):
#     def __init__(self, vocab_size, embed_dim=128, hidden_dim=128):
#         super().__init__()
#         self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
#         self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True, bidirectional=True, num_layers=1)
#         self.attention = SelfAttentionPooling(hidden_dim * 2)
#         self.classifier = nn.Sequential(
#             nn.Linear(hidden_dim * 2, hidden_dim),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(hidden_dim, 1)
#         )
        
#     def forward(self, input_ids):
#         # Input shape: [batch_size, 5, seq_len]
#         batch_size, num_choices, seq_len = input_ids.shape
#         flat_input = input_ids.view(batch_size * num_choices, seq_len)
        
#         embedded = self.embedding(flat_input)  # [batch_size * 5, seq_len, embed_dim]
#         rnn_out, _ = self.gru(embedded)        # [batch_size * 5, seq_len, hidden_dim * 2]
        
#         pooled = self.attention(rnn_out)       # [batch_size * 5, hidden_dim * 2]
#         logits = self.classifier(pooled)       # [batch_size * 5, 1]
        
#         return logits.view(batch_size, num_choices)  # [batch_size, 5]

# # ----------------------------------------------------
# # 5. Training Pipeline
# # ----------------------------------------------------
# # Load Data
# train_df = pd.read_csv(TRAIN_PATH).fillna("")
# test_df = pd.read_csv(TEST_PATH).fillna("")
# print(f"Data Loaded! Train shape: {train_df.shape}, Test shape: {test_df.shape}")

# # Fit Vocabulary
# corpus = train_df['prompt'].tolist()
# for col in ['A', 'B', 'C', 'D', 'E']:
#     corpus.extend(train_df[col].tolist())
# vocab = MCQVocabulary()
# vocab.fit(corpus)
# vocab_size = len(vocab.word2idx)
# print(f"Vocabulary fitted. Total unique tokens: {vocab_size}")

# # Stratified Train/Val split
# train_data, val_data = train_test_split(train_df, test_size=0.1, random_state=42, stratify=train_df['answer'])

# train_dataset = ScratchMCQDataset(train_data, vocab, max_len=MAX_LEN, is_train=True)
# val_dataset = ScratchMCQDataset(val_data, vocab, max_len=MAX_LEN, is_train=True)
# test_dataset = ScratchMCQDataset(test_df, vocab, max_len=MAX_LEN, is_train=False)

# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
# test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# # Initialize Model, Optimizer, Loss Function
# model = BiGRUAttentionMCQModel(vocab_size=vocab_size).to(device)
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

# # Training loop
# for epoch in range(EPOCHS):
#     model.train()
#     train_loss = 0.0
#     correct = 0
#     total = 0
    
#     for batch_x, batch_y in train_loader:
#         batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
#         optimizer.zero_grad()
#         logits = model(batch_x)
#         loss = criterion(logits, batch_y)
#         loss.backward()
#         optimizer.step()
        
#         train_loss += loss.item()
#         preds = torch.argmax(logits, dim=1)
#         correct += (preds == batch_y).sum().item()
#         total += batch_y.size(0)
        
#     train_loss /= len(train_loader)
#     train_acc = correct / total
    
#     # Validation loop
#     model.eval()
#     val_loss = 0.0
#     val_correct = 0
#     val_total = 0
#     with torch.no_grad():
#         for batch_x, batch_y in val_loader:
#             batch_x, batch_y = batch_x.to(device), batch_y.to(device)
#             logits = model(batch_x)
#             loss = criterion(logits, batch_y)
            
#             val_loss += loss.item()
#             preds = torch.argmax(logits, dim=1)
#             val_correct += (preds == batch_y).sum().item()
#             val_total += batch_y.size(0)
            
#     val_loss /= len(val_loader)
#     val_acc = val_correct / val_total
    
#     print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

# # ----------------------------------------------------
# # 6. Test Set Inference & Submission Formatting
# # ----------------------------------------------------
# print("Running inference on test set...")
# model.eval()
# test_predictions = []
# choice_letters = ['A', 'B', 'C', 'D', 'E']

# with torch.no_grad():
#     for batch_x in test_loader:
#         batch_x = batch_x.to(device)
#         logits = model(batch_x)  # [batch_size, 5]
#         probs = torch.softmax(logits, dim=1).cpu().numpy()
        
#         for p in probs:
#             # Sort option indices by probability in descending order
#             sorted_indices = np.argsort(p)[::-1][:3]
#             pred_str = " ".join([choice_letters[idx] for idx in sorted_indices])
#             test_predictions.append(pred_str)

# # Create submission DataFrame matching competition header format
# submission_df = pd.DataFrame({
#     'ID': test_df['id'],
#     'Prediction': test_predictions
# })

# # Save to submission CSV
# submission_df.to_csv(OUTPUT_PATH, index=False)
# print(f"Success! Submission file saved as '{OUTPUT_PATH}'")
# print(submission_df.head())

In [ ]:
# import os
# import numpy as np
# import pandas as pd
# import torch
# import torch.nn as nn
# from torch.utils.data import Dataset, DataLoader
# from transformers import AutoTokenizer, AutoModelForMultipleChoice
# from sklearn.model_selection import train_test_split

# # ----------------------------------------------------
# # 1. Configuration & Paths
# # ----------------------------------------------------
# TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
# TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
# OUTPUT_PATH = "submission.csv"

# MODEL_NAME = "distilbert-base-uncased"
# MAX_LEN = 128
# BATCH_SIZE = 8
# EPOCHS = 3
# LEARNING_RATE = 2e-5  # Standard learning rate for fine-tuning transformers

# # Select GPU if available
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")

# LABEL_MAP = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
# INDEX_MAP = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

# # ----------------------------------------------------
# # 2. PyTorch Dataset for Multiple Choice
# # ----------------------------------------------------
# class SimpleMCQDataset(Dataset):
#     def __init__(self, df, tokenizer, max_len=128, is_train=True):
#         self.df = df.reset_index(drop=True)
#         self.tokenizer = tokenizer
#         self.max_len = max_len
#         self.is_train = is_train
        
#     def __len__(self):
#         return len(self.df)
        
#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]
#         prompt = str(row['prompt'])
        
#         # We pair the question with each of the 5 options
#         first_sentences = [prompt] * 5
#         second_sentences = [str(row[opt]) for opt in ['A', 'B', 'C', 'D', 'E']]
        
#         # Tokenize the pairs
#         encoded = self.tokenizer(
#             first_sentences,
#             second_sentences,
#             truncation=True,
#             max_length=self.max_len,
#             padding="max_length",
#             return_tensors="pt"
#         )
        
#         # Squeeze out batch dimension from tokenizer output
#         item = {key: val.squeeze(0) for key, val in encoded.items()}
        
#         if self.is_train and 'answer' in row:
#             item['labels'] = torch.tensor(LABEL_MAP[row['answer']], dtype=torch.long)
            
#         return item

# # ----------------------------------------------------
# # 3. Model & Data Preparation
# # ----------------------------------------------------
# # Load Data
# train_df = pd.read_csv(TRAIN_PATH).fillna("")
# test_df = pd.read_csv(TEST_PATH).fillna("")
# print(f"Loaded train set: {train_df.shape}, test set: {test_df.shape}")

# # Initialize Hugging Face Tokenizer
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# # Stratified Split (90% Train, 10% Validation)
# train_data, val_data = train_test_split(train_df, test_size=0.1, random_state=42, stratify=train_df['answer'])

# train_dataset = SimpleMCQDataset(train_data, tokenizer, max_len=MAX_LEN, is_train=True)
# val_dataset = SimpleMCQDataset(val_data, tokenizer, max_len=MAX_LEN, is_train=True)
# test_dataset = SimpleMCQDataset(test_df, tokenizer, max_len=MAX_LEN, is_train=False)

# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
# test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# # Load Pre-trained Transformer Model with Multiple Choice Head
# model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME).to(device)

# # Standard AdamW Optimizer for Transformers
# optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

# # ----------------------------------------------------
# # 4. Fine-Tuning Loop
# # ----------------------------------------------------
# for epoch in range(EPOCHS):
#     model.train()
#     train_loss = 0.0
#     correct = 0
#     total = 0
    
#     for batch in train_loader:
#         input_ids = batch['input_ids'].to(device)
#         attention_mask = batch['attention_mask'].to(device)
#         labels = batch['labels'].to(device)
        
#         optimizer.zero_grad()
        
#         # Forward pass through model (calculates loss internally if labels are passed)
#         outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
#         loss = outputs.loss
#         logits = outputs.logits
        
#         # Backward pass & weight update
#         loss.backward()
#         optimizer.step()
        
#         train_loss += loss.item()
#         preds = torch.argmax(logits, dim=1)
#         correct += (preds == labels).sum().item()
#         total += labels.size(0)
        
#     train_loss /= len(train_loader)
#     train_acc = correct / total
    
#     # Validation Loop
#     model.eval()
#     val_loss = 0.0
#     val_correct = 0
#     val_total = 0
#     with torch.no_grad():
#         for batch in val_loader:
#             input_ids = batch['input_ids'].to(device)
#             attention_mask = batch['attention_mask'].to(device)
#             labels = batch['labels'].to(device)
            
#             outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
#             loss = outputs.loss
#             logits = outputs.logits
            
#             val_loss += loss.item()
#             preds = torch.argmax(logits, dim=1)
#             val_correct += (preds == labels).sum().item()
#             val_total += labels.size(0)
            
#     val_loss /= len(val_loader)
#     val_acc = val_correct / val_total
    
#     print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

# # Save the trained model checkpoint
# model.save_pretrained("distilbert_mcq_model")
# tokenizer.save_pretrained("distilbert_mcq_model")
# print("Model saved successfully!")

# # ----------------------------------------------------
# # 5. Inference & Submission Generation
# # ----------------------------------------------------
# print("Running inference on test set...")
# model.eval()
# test_predictions = []

# with torch.no_grad():
#     for batch in test_loader:
#         input_ids = batch['input_ids'].to(device)
#         attention_mask = batch['attention_mask'].to(device)
        
#         outputs = model(input_ids=input_ids, attention_mask=attention_mask)
#         logits = outputs.logits
#         probs = torch.softmax(logits, dim=1).cpu().numpy()
        
#         for p in probs:
#             # Get the indices of the top 3 options sorted by highest probability
#             sorted_indices = np.argsort(p)[::-1][:3]
#             pred_str = " ".join([INDEX_MAP[idx] for idx in sorted_indices])
#             test_predictions.append(pred_str)

# # Save test predictions in required competition format
# submission_df = pd.DataFrame({
#     'ID': test_df['id'],
#     'Prediction': test_predictions
# })

# submission_df.to_csv(OUTPUT_PATH, index=False)
# print(f"Success! Submission file saved as '{OUTPUT_PATH}'")
# print(submission_df.head())

In [ ]:
# import os
# import numpy as np
# import pandas as pd
# import torch
# import torch.nn as nn
# from torch.utils.data import Dataset, DataLoader
# from transformers import AutoTokenizer, AutoModelForMultipleChoice
# from sklearn.model_selection import train_test_split

# # ----------------------------------------------------
# # 1. Configuration & Paths
# # ----------------------------------------------------
# TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
# TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
# OUTPUT_PATH = "submission.csv"

# # Using the standard BERT-base-uncased model
# MODEL_NAME = "bert-base-uncased"
# MAX_LEN = 128
# BATCH_SIZE = 8
# EPOCHS = 3
# LEARNING_RATE = 2e-5

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")

# LABEL_MAP = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
# INDEX_MAP = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

# # ----------------------------------------------------
# # 2. PyTorch Dataset for MCQ
# # ----------------------------------------------------
# class MCQDataset(Dataset):
#     def __init__(self, df, tokenizer, max_len=128, is_train=True):
#         self.df = df.reset_index(drop=True)
#         self.tokenizer = tokenizer
#         self.max_len = max_len
#         self.is_train = is_train
        
#     def __len__(self):
#         return len(self.df)
        
#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]
#         prompt = str(row['prompt'])
        
#         # We pair the question with each of the 5 options
#         first_sentences = [prompt] * 5
#         second_sentences = [str(row[opt]) for opt in ['A', 'B', 'C', 'D', 'E']]
        
#         # Tokenize the pairs
#         encoded = self.tokenizer(
#             first_sentences,
#             second_sentences,
#             truncation=True,
#             max_length=self.max_len,
#             padding="max_length",
#             return_tensors="pt"
#         )
        
#         item = {key: val.squeeze(0) for key, val in encoded.items()}
        
#         if self.is_train and 'answer' in row:
#             item['labels'] = torch.tensor(LABEL_MAP[row['answer']], dtype=torch.long)
            
#         return item

# # ----------------------------------------------------
# # 3. Model & Data Preparation
# # ----------------------------------------------------
# train_df = pd.read_csv(TRAIN_PATH).fillna("")
# test_df = pd.read_csv(TEST_PATH).fillna("")
# print(f"Loaded train set: {train_df.shape}, test set: {test_df.shape}")

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# train_data, val_data = train_test_split(train_df, test_size=0.1, random_state=42, stratify=train_df['answer'])

# train_dataset = MCQDataset(train_data, tokenizer, max_len=MAX_LEN, is_train=True)
# val_dataset = MCQDataset(val_data, tokenizer, max_len=MAX_LEN, is_train=True)
# test_dataset = MCQDataset(test_df, tokenizer, max_len=MAX_LEN, is_train=False)

# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
# test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# # Load Pre-trained BERT Model for Multiple Choice classification
# model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME).to(device)

# optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

# # ----------------------------------------------------
# # 4. Training Loop
# # ----------------------------------------------------
# for epoch in range(EPOCHS):
#     model.train()
#     train_loss = 0.0
#     correct = 0
#     total = 0
    
#     for batch in train_loader:
#         input_ids = batch['input_ids'].to(device)
#         attention_mask = batch['attention_mask'].to(device)
#         # BERT requires token_type_ids to distinguish prompt and option segments
#         token_type_ids = batch['token_type_ids'].to(device)
#         labels = batch['labels'].to(device)
        
#         optimizer.zero_grad()
        
#         outputs = model(
#             input_ids=input_ids, 
#             attention_mask=attention_mask, 
#             token_type_ids=token_type_ids, 
#             labels=labels
#         )
#         loss = outputs.loss
#         logits = outputs.logits
        
#         loss.backward()
#         optimizer.step()
        
#         train_loss += loss.item()
#         preds = torch.argmax(logits, dim=1)
#         correct += (preds == labels).sum().item()
#         total += labels.size(0)
        
#     train_loss /= len(train_loader)
#     train_acc = correct / total
    
#     # Validation Loop
#     model.eval()
#     val_loss = 0.0
#     val_correct = 0
#     val_total = 0
#     with torch.no_grad():
#         for batch in val_loader:
#             input_ids = batch['input_ids'].to(device)
#             attention_mask = batch['attention_mask'].to(device)
#             token_type_ids = batch['token_type_ids'].to(device)
#             labels = batch['labels'].to(device)
            
#             outputs = model(
#                 input_ids=input_ids, 
#                 attention_mask=attention_mask, 
#                 token_type_ids=token_type_ids, 
#                 labels=labels
#             )
#             loss = outputs.loss
#             logits = outputs.logits
            
#             val_loss += loss.item()
#             preds = torch.argmax(logits, dim=1)
#             val_correct += (preds == labels).sum().item()
#             val_total += labels.size(0)
            
#     val_loss /= len(val_loader)
#     val_acc = val_correct / val_total
    
#     print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

# # Save the trained model checkpoint
# model.save_pretrained("bert_mcq_model")
# tokenizer.save_pretrained("bert_mcq_model")
# print("BERT model saved successfully!")

# # ----------------------------------------------------
# # 5. Inference & Submission Generation
# # ----------------------------------------------------
# print("Running inference on test set...")
# model.eval()
# test_predictions = []

# with torch.no_grad():
#     for batch in test_loader:
#         input_ids = batch['input_ids'].to(device)
#         attention_mask = batch['attention_mask'].to(device)
#         token_type_ids = batch['token_type_ids'].to(device)
        
#         outputs = model(
#             input_ids=input_ids, 
#             attention_mask=attention_mask, 
#             token_type_ids=token_type_ids
#         )
#         logits = outputs.logits
#         probs = torch.softmax(logits, dim=1).cpu().numpy()
        
#         for p in probs:
#             sorted_indices = np.argsort(p)[::-1][:3]
#             pred_str = " ".join([INDEX_MAP[idx] for idx in sorted_indices])
#             test_predictions.append(pred_str)

# # Save test predictions to submission CSV
# submission_df = pd.DataFrame({
#     'ID': test_df['id'],
#     'Prediction': test_predictions
# })

# submission_df.to_csv(OUTPUT_PATH, index=False)
# print(f"Success! Submission file saved as '{OUTPUT_PATH}'")
# print(submission_df.head())

In [ ]:
# import os
# import re
# import numpy as np
# import pandas as pd
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import Dataset, DataLoader
# from sklearn.model_selection import train_test_split

# # ----------------------------------------------------
# # 1. Hyperparameters & Configuration
# # ----------------------------------------------------
# TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
# TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
# OUTPUT_PATH = "submission.csv"

# MAX_LEN = 192         # Prevents truncating options D and E
# BATCH_SIZE = 32
# EPOCHS = 10
# LEARNING_RATE = 0.001
# EMBED_DIM = 256
# HIDDEN_DIM = 256
# NUM_LAYERS = 2
# DROPOUT_RATE = 0.4    # Increased dropout to force generalization and prevent memorization

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")

# PAD_TOKEN = "<PAD>"
# UNK_TOKEN = "<UNK>"
# SEP_TOKEN = "<SEP>"

# # ----------------------------------------------------
# # 2. Text Preprocessing & Vocabulary Build
# # ----------------------------------------------------
# def clean_text(text):
#     if not isinstance(text, str):
#         return ""
#     text = text.lower()
#     text = re.sub(r"[^\w\s\-\.]", "", text)
#     text = re.sub(r"\s+", " ", text).strip()
#     return text

# class MCQVocabulary:
#     def __init__(self, max_vocab_size=20000):
#         self.max_vocab_size = max_vocab_size
#         self.word2idx = {PAD_TOKEN: 0, UNK_TOKEN: 1, SEP_TOKEN: 2}
#         self.idx2word = {0: PAD_TOKEN, 1: UNK_TOKEN, 2: SEP_TOKEN}
        
#     def fit(self, texts):
#         word_counts = {}
#         for text in texts:
#             cleaned = clean_text(text)
#             for word in cleaned.split():
#                 word_counts[word] = word_counts.get(word, 0) + 1
                
#         sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
#         for word, count in sorted_words[:self.max_vocab_size]:
#             if word not in self.word2idx:
#                 idx = len(self.word2idx)
#                 self.word2idx[word] = idx
#                 self.idx2word[idx] = word
                
#     def encode(self, text, max_len=128):
#         cleaned = clean_text(text)
#         tokens = cleaned.split()
#         encoded = [self.word2idx.get(w, self.word2idx[UNK_TOKEN]) for w in tokens]
        
#         if len(encoded) > max_len:
#             encoded = encoded[:max_len]
#         else:
#             encoded = encoded + [self.word2idx[PAD_TOKEN]] * (max_len - len(encoded))
#         return encoded

# # ----------------------------------------------------
# # 3. Dataset Setup
# # ----------------------------------------------------
# class ScratchMCQDataset(Dataset):
#     def __init__(self, df, vocab, max_len=128, is_train=True):
#         self.df = df.reset_index(drop=True)
#         self.vocab = vocab
#         self.max_len = max_len
#         self.is_train = is_train
#         self.label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
        
#     def __len__(self):
#         return len(self.df)
        
#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]
#         prompt = str(row['prompt'])
        
#         input_ids = []
#         for choice in ['A', 'B', 'C', 'D', 'E']:
#             choice_text = str(row[choice])
#             combined_text = f"{prompt} {SEP_TOKEN} {choice_text}"
#             encoded = self.vocab.encode(combined_text, max_len=self.max_len)
#             input_ids.append(encoded)
            
#         input_ids = torch.tensor(input_ids, dtype=torch.long)
        
#         if self.is_train and 'answer' in row:
#             label = torch.tensor(self.label_map[row['answer']], dtype=torch.long)
#             return input_ids, label
#         return input_ids

# # ----------------------------------------------------
# # 4. Tuned Model Architecture
# # ----------------------------------------------------
# class SelfAttentionPooling(nn.Module):
#     def __init__(self, hidden_dim):
#         super().__init__()
#         self.attention = nn.Sequential(
#             nn.Linear(hidden_dim, hidden_dim // 2),
#             nn.Tanh(),
#             nn.Linear(hidden_dim // 2, 1)
#         )
        
#     def forward(self, rnn_outputs):
#         weights = self.attention(rnn_outputs)
#         weights = torch.softmax(weights, dim=1)
#         pooled = torch.sum(rnn_outputs * weights, dim=1)
#         return pooled

# class TunedBiGRUAttentionMCQModel(nn.Module):
#     def __init__(self, vocab_size, embed_dim=256, hidden_dim=256, num_layers=2, dropout=0.4):
#         super().__init__()
#         self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
#         self.gru = nn.GRU(
#             embed_dim, 
#             hidden_dim, 
#             batch_first=True, 
#             bidirectional=True, 
#             num_layers=num_layers,
#             dropout=dropout if num_layers > 1 else 0.0
#         )
#         self.attention = SelfAttentionPooling(hidden_dim * 2)
#         self.classifier = nn.Sequential(
#             nn.Linear(hidden_dim * 2, hidden_dim),
#             nn.LayerNorm(hidden_dim),
#             nn.ReLU(),
#             nn.Dropout(dropout),
#             nn.Linear(hidden_dim, 1)
#         )
        
#     def forward(self, input_ids):
#         batch_size, num_choices, seq_len = input_ids.shape
#         flat_input = input_ids.view(batch_size * num_choices, seq_len)
        
#         embedded = self.embedding(flat_input)
#         rnn_out, _ = self.gru(embedded)
        
#         pooled = self.attention(rnn_out)
#         logits = self.classifier(pooled)
        
#         return logits.view(batch_size, num_choices)

# # ----------------------------------------------------
# # 5. Training Pipeline with Split Leakage Protection
# # ----------------------------------------------------
# train_raw = pd.read_csv(TRAIN_PATH).fillna("")
# test_df = pd.read_csv(TEST_PATH).fillna("")
# print(f"Original Train shape: {train_raw.shape}, Test shape: {test_df.shape}")

# # Fit Vocabulary on both train and test to prevent OOV issues on the test set
# print("Fitting vocabulary on train & test sets...")
# corpus = train_raw['prompt'].tolist() + test_df['prompt'].tolist()
# for col in ['A', 'B', 'C', 'D', 'E']:
#     corpus.extend(train_raw[col].tolist())
#     corpus.extend(test_df[col].tolist())
# vocab = MCQVocabulary()
# vocab.fit(corpus)
# vocab_size = len(vocab.word2idx)
# print(f"Vocabulary Size: {vocab_size}")

# # Protect against validation split leakage: deduplicate train set before splitting
# # This gives a real representation of how the model performs on unseen questions.
# train_df = train_raw.drop_duplicates(subset=['prompt', 'A', 'B', 'C', 'D', 'E']).reset_index(drop=True)
# print(f"Deduplicated Train shape: {train_df.shape} (Removed duplicate questions)")

# # Stratified Split (90% Train, 10% Validation)
# train_data, val_data = train_test_split(train_df, test_size=0.1, random_state=42, stratify=train_df['answer'])

# train_dataset = ScratchMCQDataset(train_data, vocab, max_len=MAX_LEN, is_train=True)
# val_dataset = ScratchMCQDataset(val_data, vocab, max_len=MAX_LEN, is_train=True)
# test_dataset = ScratchMCQDataset(test_df, vocab, max_len=MAX_LEN, is_train=False)

# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
# test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# # Initialize Model, Optimizer, Scheduler, Loss
# model = TunedBiGRUAttentionMCQModel(
#     vocab_size=vocab_size, 
#     embed_dim=EMBED_DIM, 
#     hidden_dim=HIDDEN_DIM, 
#     num_layers=NUM_LAYERS,
#     dropout=DROPOUT_RATE
# ).to(device)

# criterion = nn.CrossEntropyLoss()
# optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
# scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# # Training loop
# for epoch in range(EPOCHS):
#     model.train()
#     train_loss = 0.0
#     correct = 0
#     total = 0
    
#     for batch_x, batch_y in train_loader:
#         batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
#         optimizer.zero_grad()
#         logits = model(batch_x)
#         loss = criterion(logits, batch_y)
#         loss.backward()
        
#         nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#         optimizer.step()
        
#         train_loss += loss.item()
#         preds = torch.argmax(logits, dim=1)
#         correct += (preds == batch_y).sum().item()
#         total += batch_y.size(0)
        
#     scheduler.step()
#     train_loss /= len(train_loader)
#     train_acc = correct / total
    
#     # Validation loop
#     model.eval()
#     val_loss = 0.0
#     val_correct = 0
#     val_total = 0
#     with torch.no_grad():
#         for batch_x, batch_y in val_loader:
#             batch_x, batch_y = batch_x.to(device), batch_y.to(device)
#             logits = model(batch_x)
#             loss = criterion(logits, batch_y)
            
#             val_loss += loss.item()
#             preds = torch.argmax(logits, dim=1)
#             val_correct += (preds == batch_y).sum().item()
#             val_total += batch_y.size(0)
            
#     val_loss /= len(val_loader)
#     val_acc = val_correct / val_total
    
#     current_lr = optimizer.param_groups[0]['lr']
#     print(f"Epoch {epoch+1}/{EPOCHS} | LR: {current_lr:.6f} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

# # ----------------------------------------------------
# # 6. Test Set Inference & Submission Formatting
# # ----------------------------------------------------
# print("Running inference on test set...")
# model.eval()
# test_predictions = []
# choice_letters = ['A', 'B', 'C', 'D', 'E']

# with torch.no_grad():
#     for batch_x in test_loader:
#         batch_x = batch_x.to(device)
#         logits = model(batch_x)
#         probs = torch.softmax(logits, dim=1).cpu().numpy()
        
#         for p in probs:
#             sorted_indices = np.argsort(p)[::-1][:3]
#             pred_str = " ".join([choice_letters[idx] for idx in sorted_indices])
#             test_predictions.append(pred_str)

# # Create submission DataFrame
# submission_df = pd.DataFrame({
#     'ID': test_df['id'],
#     'Prediction': test_predictions
# })

# # Save to submission CSV
# submission_df.to_csv(OUTPUT_PATH, index=False)
# print(f"Success! Submission file saved as '{OUTPUT_PATH}'")

NN from scratch

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# ----------------------------------------------------
# 1. Configuration & Hyperparameters
# ----------------------------------------------------
TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
OUTPUT_PATH = "submission.csv"

MAX_LEN = 256         # Increased to 256 to ensure no scientific context is cut off
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001
EMBED_DIM = 256
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT_RATE = 0.4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
SEP_TOKEN = "<SEP>"

# Set up seeding helper
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ----------------------------------------------------
# 2. Rich Text Preprocessing (Preserves Equations)
# ----------------------------------------------------
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    # Normalize whitespaces but preserve punctuation and mathematical symbols (e.g. +, =, ρ̂)
    text = re.sub(r"\s+", " ", text).strip()
    return text

class MCQVocabulary:
    def __init__(self, max_vocab_size=20000):
        self.max_vocab_size = max_vocab_size
        self.word2idx = {PAD_TOKEN: 0, UNK_TOKEN: 1, SEP_TOKEN: 2}
        self.idx2word = {0: PAD_TOKEN, 1: UNK_TOKEN, 2: SEP_TOKEN}
        
    def fit(self, texts):
        word_counts = {}
        for text in texts:
            cleaned = clean_text(text)
            for word in cleaned.split():
                word_counts[word] = word_counts.get(word, 0) + 1
                
        sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
        for word, count in sorted_words[:self.max_vocab_size]:
            if word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word
                
    def encode(self, text, max_len=128):
        cleaned = clean_text(text)
        tokens = cleaned.split()
        encoded = [self.word2idx.get(w, self.word2idx[UNK_TOKEN]) for w in tokens]
        
        if len(encoded) > max_len:
            encoded = encoded[:max_len]
        else:
            encoded = encoded + [self.word2idx[PAD_TOKEN]] * (max_len - len(encoded))
        return encoded

# ----------------------------------------------------
# 3. Dataset Implementation
# ----------------------------------------------------
class ScratchMCQDataset(Dataset):
    def __init__(self, df, vocab, max_len=128, is_train=True):
        self.df = df.reset_index(drop=True)
        self.vocab = vocab
        self.max_len = max_len
        self.is_train = is_train
        self.label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt'])
        
        input_ids = []
        for choice in ['A', 'B', 'C', 'D', 'E']:
            choice_text = str(row[choice])
            combined_text = f"{prompt} {SEP_TOKEN} {choice_text}"
            encoded = self.vocab.encode(combined_text, max_len=self.max_len)
            input_ids.append(encoded)
            
        input_ids = torch.tensor(input_ids, dtype=torch.long)
        
        if self.is_train and 'answer' in row:
            label = torch.tensor(self.label_map[row['answer']], dtype=torch.long)
            return input_ids, label
        return input_ids

# ----------------------------------------------------
# 4. Neural Network Architecture from Scratch
# ----------------------------------------------------
class SelfAttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )
        
    def forward(self, rnn_outputs):
        weights = self.attention(rnn_outputs)
        weights = torch.softmax(weights, dim=1)
        pooled = torch.sum(rnn_outputs * weights, dim=1)
        return pooled

class TunedBiGRUAttentionMCQModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=256, num_layers=2, dropout=0.4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(
            embed_dim, 
            hidden_dim, 
            batch_first=True, 
            bidirectional=True, 
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.attention = SelfAttentionPooling(hidden_dim * 2)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )
        
    def forward(self, input_ids):
        batch_size, num_choices, seq_len = input_ids.shape
        flat_input = input_ids.view(batch_size * num_choices, seq_len)
        
        embedded = self.embedding(flat_input)
        rnn_out, _ = self.gru(embedded)
        
        pooled = self.attention(rnn_out)
        logits = self.classifier(pooled)
        
        return logits.view(batch_size, num_choices)

# ----------------------------------------------------
# 5. Data Prep & Vocabulary Building
# ----------------------------------------------------
train_raw = pd.read_csv(TRAIN_PATH).fillna("")
test_df = pd.read_csv(TEST_PATH).fillna("")
print(f"Loaded Raw datasets | Train: {train_raw.shape}, Test: {test_df.shape}")

# Fit Vocabulary on both train and test to prevent OOV issues
print("Fitting vocabulary on train & test sets...")
corpus = train_raw['prompt'].tolist() + test_df['prompt'].tolist()
for col in ['A', 'B', 'C', 'D', 'E']:
    corpus.extend(train_raw[col].tolist())
    corpus.extend(test_df[col].tolist())
vocab = MCQVocabulary()
vocab.fit(corpus)
vocab_size = len(vocab.word2idx)
print(f"Vocabulary Size: {vocab_size}")

# Protect validation split from leakage: deduplicate train set before split
train_df = train_raw.drop_duplicates(subset=['prompt', 'A', 'B', 'C', 'D', 'E']).reset_index(drop=True)
print(f"Deduplicated Train shape: {train_df.shape}")

# Split splits
train_data, val_data = train_test_split(train_df, test_size=0.1, random_state=42, stratify=train_df['answer'])

train_dataset = ScratchMCQDataset(train_data, vocab, max_len=MAX_LEN, is_train=True)
val_dataset = ScratchMCQDataset(val_data, vocab, max_len=MAX_LEN, is_train=True)
test_dataset = ScratchMCQDataset(test_df, vocab, max_len=MAX_LEN, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ----------------------------------------------------
# 6. Train Models Function (Multi-seed)
# ----------------------------------------------------
def train_model(seed):
    print(f"\n--- Training Model with Seed {seed} ---")
    set_seed(seed)
    
    model = TunedBiGRUAttentionMCQModel(
        vocab_size=vocab_size, 
        embed_dim=EMBED_DIM, 
        hidden_dim=HIDDEN_DIM, 
        num_layers=NUM_LAYERS,
        dropout=DROPOUT_RATE
    ).to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        correct = 0
        total = 0
        
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            correct += (preds == batch_y).sum().item()
            total += batch_y.size(0)
            
        scheduler.step()
        train_loss /= len(train_loader)
        train_acc = correct / total
        
        # Validation
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                logits = model(batch_x)
                loss = criterion(logits, batch_y)
                
                val_loss += loss.item()
                preds = torch.argmax(logits, dim=1)
                val_correct += (preds == batch_y).sum().item()
                val_total += batch_y.size(0)
                
        val_loss /= len(val_loader)
        val_acc = val_correct / val_total
        
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        
    return model

# Train Model A (Seed 42)
model_a = train_model(seed=42)

# Train Model B (Seed 2026)
model_b = train_model(seed=2026)

# ----------------------------------------------------
# 7. Ensembled Inference on Test Set
# ----------------------------------------------------
print("\nRunning ensembled inference on test set...")
model_a.eval()
model_b.eval()
test_predictions = []
choice_letters = ['A', 'B', 'C', 'D', 'E']

with torch.no_grad():
    for batch_x in test_loader:
        batch_x = batch_x.to(device)
        
        # Get logits from both models
        logits_a = model_a(batch_x)
        logits_b = model_b(batch_x)
        
        # Convert to probabilities
        probs_a = torch.softmax(logits_a, dim=1).cpu().numpy()
        probs_b = torch.softmax(logits_b, dim=1).cpu().numpy()
        
        # Average the probabilities
        probs_ensembled = 0.5 * probs_a + 0.5 * probs_b
        
        for p in probs_ensembled:
            sorted_indices = np.argsort(p)[::-1][:3]
            pred_str = " ".join([choice_letters[idx] for idx in sorted_indices])
            test_predictions.append(pred_str)

# Save predictions to submission CSV
submission_df = pd.DataFrame({
    'ID': test_df['id'],
    'Prediction': test_predictions
})

submission_df.to_csv(OUTPUT_PATH, index=False)
print(f"Success! Ensembled submission file saved to '{OUTPUT_PATH}'")

Model of the choice

In [ ]:
# import os
# import re
# import numpy as np
# import pandas as pd
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import Dataset, DataLoader
# from transformers import AutoTokenizer, AutoModelForMultipleChoice
# from sklearn.model_selection import train_test_split

# # ----------------------------------------------------
# # 1. Configuration & Paths
# # ----------------------------------------------------
# TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
# TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
# OUTPUT_PATH = "submission.csv"

# # Seeding for reproducibility
# def set_seed(seed=42):
#     torch.manual_seed(seed)
#     torch.cuda.manual_seed_all(seed)
#     np.random.seed(seed)
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False

# set_seed(42)

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")

# # Label mappings
# LABEL_MAP = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
# INDEX_MAP = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

# # ----------------------------------------------------
# # 2. Text Preprocessing & Cleaning (Punctuation Stripped)
# # ----------------------------------------------------
# # We restore the clean tokenization from V7 to prevent vocabulary pollution
# def clean_text_scratch(text):
#     if not isinstance(text, str):
#         return ""
#     text = text.lower()
#     text = re.sub(r"[^\w\s\-\.]", "", text)  # strips punctuations that pollute vocab
#     text = re.sub(r"\s+", " ", text).strip()
#     return text

# class MCQVocabulary:
#     def __init__(self, max_vocab_size=20000):
#         self.max_vocab_size = max_vocab_size
#         self.word2idx = {"<PAD>": 0, "<UNK>": 1, "<SEP>": 2}
#         self.idx2word = {0: "<PAD>", 1: "<UNK>", 2: "<SEP>"}
        
#     def fit(self, texts):
#         word_counts = {}
#         for text in texts:
#             cleaned = clean_text_scratch(text)
#             for word in cleaned.split():
#                 word_counts[word] = word_counts.get(word, 0) + 1
                
#         sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
#         for word, count in sorted_words[:self.max_vocab_size]:
#             if word not in self.word2idx:
#                 idx = len(self.word2idx)
#                 self.word2idx[word] = idx
#                 self.idx2word[idx] = word
                
#     def encode(self, text, max_len=128):
#         cleaned = clean_text_scratch(text)
#         tokens = cleaned.split()
#         encoded = [self.word2idx.get(w, self.word2idx["<UNK>"]) for w in tokens]
        
#         if len(encoded) > max_len:
#             encoded = encoded[:max_len]
#         else:
#             encoded = encoded + [self.word2idx["<PAD>"]] * (max_len - len(encoded))
#         return encoded

# # ----------------------------------------------------
# # 3. Model 1 (Scratch BiGRU) Dataset & Model
# # ----------------------------------------------------
# class ScratchMCQDataset(Dataset):
#     def __init__(self, df, vocab, max_len=192, is_train=True):
#         self.df = df.reset_index(drop=True)
#         self.vocab = vocab
#         self.max_len = max_len
#         self.is_train = is_train
        
#     def __len__(self):
#         return len(self.df)
        
#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]
#         prompt = str(row['prompt'])
        
#         input_ids = []
#         for choice in ['A', 'B', 'C', 'D', 'E']:
#             choice_text = str(row[choice])
#             combined_text = f"{prompt} <SEP> {choice_text}"
#             encoded = self.vocab.encode(combined_text, max_len=self.max_len)
#             input_ids.append(encoded)
            
#         input_ids = torch.tensor(input_ids, dtype=torch.long)
        
#         if self.is_train and 'answer' in row:
#             label = torch.tensor(LABEL_MAP[row['answer']], dtype=torch.long)
#             return input_ids, label
#         return input_ids

# class SelfAttentionPooling(nn.Module):
#     def __init__(self, hidden_dim):
#         super().__init__()
#         self.attention = nn.Sequential(
#             nn.Linear(hidden_dim, hidden_dim // 2),
#             nn.Tanh(),
#             nn.Linear(hidden_dim // 2, 1)
#         )
#     def forward(self, rnn_outputs):
#         weights = self.attention(rnn_outputs)
#         weights = torch.softmax(weights, dim=1)
#         return torch.sum(rnn_outputs * weights, dim=1)

# class TunedBiGRUAttentionMCQModel(nn.Module):
#     def __init__(self, vocab_size, embed_dim=256, hidden_dim=256, num_layers=2, dropout=0.4):
#         super().__init__()
#         self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
#         self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True, bidirectional=True, num_layers=num_layers, dropout=dropout)
#         self.attention = SelfAttentionPooling(hidden_dim * 2)
#         self.classifier = nn.Sequential(
#             nn.Linear(hidden_dim * 2, hidden_dim),
#             nn.LayerNorm(hidden_dim),
#             nn.ReLU(),
#             nn.Dropout(dropout),
#             nn.Linear(hidden_dim, 1)
#         )
#     def forward(self, input_ids):
#         batch_size, num_choices, seq_len = input_ids.shape
#         flat_input = input_ids.view(batch_size * num_choices, seq_len)
#         embedded = self.embedding(flat_input)
#         rnn_out, _ = self.gru(embedded)
#         pooled = self.attention(rnn_out)
#         logits = self.classifier(pooled)
#         return logits.view(batch_size, num_choices)

# # ----------------------------------------------------
# # 4. Model 2 (Pretrained DistilBERT) Dataset
# # ----------------------------------------------------
# class SimpleMCQDataset(Dataset):
#     def __init__(self, df, tokenizer, max_len=128, is_train=True):
#         self.df = df.reset_index(drop=True)
#         self.tokenizer = tokenizer
#         self.max_len = max_len
#         self.is_train = is_train
        
#     def __len__(self):
#         return len(self.df)
        
#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]
#         prompt = str(row['prompt'])
        
#         first_sentences = [prompt] * 5
#         second_sentences = [str(row[opt]) for opt in ['A', 'B', 'C', 'D', 'E']]
        
#         encoded = self.tokenizer(
#             first_sentences,
#             second_sentences,
#             truncation=True,
#             max_length=self.max_len,
#             padding="max_length",
#             return_tensors="pt"
#         )
        
#         item = {key: val.squeeze(0) for key, val in encoded.items()}
#         if self.is_train and 'answer' in row:
#             item['labels'] = torch.tensor(LABEL_MAP[row['answer']], dtype=torch.long)
#         return item

# # ----------------------------------------------------
# # 5. Data Prep & Vocab fitting
# # ----------------------------------------------------
# train_raw = pd.read_csv(TRAIN_PATH).fillna("")
# test_df = pd.read_csv(TEST_PATH).fillna("")
# print(f"Data Loaded. Train: {train_raw.shape}, Test: {test_df.shape}")

# # Deduplicate to prevent validation split leakage
# train_df = train_raw.drop_duplicates(subset=['prompt', 'A', 'B', 'C', 'D', 'E']).reset_index(drop=True)
# print(f"Deduplicated Train shape: {train_df.shape}")

# train_data, val_data = train_test_split(train_df, test_size=0.1, random_state=42, stratify=train_df['answer'])

# # Build Vocab for Model 1 on both Train and Test to prevent test-set OOV issues
# print("Fitting Scratch Vocabulary...")
# corpus = train_raw['prompt'].tolist() + test_df['prompt'].tolist()
# for col in ['A', 'B', 'C', 'D', 'E']:
#     corpus.extend(train_raw[col].tolist())
#     corpus.extend(test_df[col].tolist())
# vocab = MCQVocabulary()
# vocab.fit(corpus)
# vocab_size = len(vocab.word2idx)
# print(f"Vocab size: {vocab_size}")

# # ----------------------------------------------------
# # 6. Training Model 1: Scratch Model
# # ----------------------------------------------------
# print("\n--- Training Model 1: Custom Scratch Model ---")
# train_dataset_scratch = ScratchMCQDataset(train_data, vocab, max_len=192, is_train=True)
# val_dataset_scratch = ScratchMCQDataset(val_data, vocab, max_len=192, is_train=True)
# test_dataset_scratch = ScratchMCQDataset(test_df, vocab, max_len=192, is_train=False)

# train_loader_scratch = DataLoader(train_dataset_scratch, batch_size=32, shuffle=True)
# val_loader_scratch = DataLoader(val_dataset_scratch, batch_size=32, shuffle=False)
# test_loader_scratch = DataLoader(test_dataset_scratch, batch_size=32, shuffle=False)

# model_scratch = TunedBiGRUAttentionMCQModel(vocab_size=vocab_size).to(device)
# criterion_scratch = nn.CrossEntropyLoss()
# optimizer_scratch = optim.AdamW(model_scratch.parameters(), lr=0.001, weight_decay=1e-4)
# scheduler_scratch = optim.lr_scheduler.CosineAnnealingLR(optimizer_scratch, T_max=10, eta_min=1e-6)

# for epoch in range(10):
#     model_scratch.train()
#     train_loss = 0.0
#     correct = 0
#     total = 0
#     for batch_x, batch_y in train_loader_scratch:
#         batch_x, batch_y = batch_x.to(device), batch_y.to(device)
#         optimizer_scratch.zero_grad()
#         logits = model_scratch(batch_x)
#         loss = criterion_scratch(logits, batch_y)
#         loss.backward()
#         nn.utils.clip_grad_norm_(model_scratch.parameters(), max_norm=1.0)
#         optimizer_scratch.step()
        
#         train_loss += loss.item()
#         correct += (torch.argmax(logits, dim=1) == batch_y).sum().item()
#         total += batch_y.size(0)
#     scheduler_scratch.step()
    
#     # Val
#     model_scratch.eval()
#     val_correct = 0
#     val_total = 0
#     with torch.no_grad():
#         for batch_x, batch_y in val_loader_scratch:
#             batch_x, batch_y = batch_x.to(device), batch_y.to(device)
#             logits = model_scratch(batch_x)
#             val_correct += (torch.argmax(logits, dim=1) == batch_y).sum().item()
#             val_total += batch_y.size(0)
#     print(f"Epoch {epoch+1}/10 | Train Loss: {train_loss/len(train_loader_scratch):.4f} | Train Acc: {correct/total:.4f} | Val Acc: {val_correct/val_total:.4f}")

# # ----------------------------------------------------
# # 7. Training Model 2: DistilBERT Model
# # ----------------------------------------------------
# print("\n--- Training Model 2: Pre-trained DistilBERT ---")
# tokenizer_db = AutoTokenizer.from_pretrained("distilbert-base-uncased")
# train_dataset_db = SimpleMCQDataset(train_data, tokenizer_db, max_len=128, is_train=True)
# val_dataset_db = SimpleMCQDataset(val_data, tokenizer_db, max_len=128, is_train=True)
# test_dataset_db = SimpleMCQDataset(test_df, tokenizer_db, max_len=128, is_train=False)

# train_loader_db = DataLoader(train_dataset_db, batch_size=8, shuffle=True)
# val_loader_db = DataLoader(val_dataset_db, batch_size=8, shuffle=False)
# test_loader_db = DataLoader(test_dataset_db, batch_size=8, shuffle=False)

# model_db = AutoModelForMultipleChoice.from_pretrained("distilbert-base-uncased").to(device)
# optimizer_db = optim.AdamW(model_db.parameters(), lr=2e-5, weight_decay=0.01)

# for epoch in range(3):
#     model_db.train()
#     train_loss = 0.0
#     correct = 0
#     total = 0
#     for batch in train_loader_db:
#         input_ids = batch['input_ids'].to(device)
#         attention_mask = batch['attention_mask'].to(device)
#         labels = batch['labels'].to(device)
        
#         optimizer_db.zero_grad()
#         outputs = model_db(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
#         loss = outputs.loss
#         logits = outputs.logits
        
#         loss.backward()
#         optimizer_db.step()
        
#         train_loss += loss.item()
#         correct += (torch.argmax(logits, dim=1) == labels).sum().item()
#         total += labels.size(0)
        
#     model_db.eval()
#     val_correct = 0
#     val_total = 0
#     with torch.no_grad():
#         for batch in val_loader_db:
#             input_ids = batch['input_ids'].to(device)
#             attention_mask = batch['attention_mask'].to(device)
#             labels = batch['labels'].to(device)
#             outputs = model_db(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
#             val_correct += (torch.argmax(outputs.logits, dim=1) == labels).sum().item()
#             val_total += labels.size(0)
#     print(f"Epoch {epoch+1}/3 | Train Loss: {train_loss/len(train_loader_db):.4f} | Train Acc: {correct/total:.4f} | Val Acc: {val_correct/val_total:.4f}")

# # ----------------------------------------------------
# # 8. Ensembled Inference on Test Set
# # ----------------------------------------------------
# print("\nRunning ensembled inference (30% Scratch + 70% DistilBERT)...")
# model_scratch.eval()
# model_db.eval()

# probs_scratch_list = []
# with torch.no_grad():
#     for batch_x in test_loader_scratch:
#         batch_x = batch_x.to(device)
#         logits = model_scratch(batch_x)
#         probs_scratch_list.append(torch.softmax(logits, dim=1).cpu().numpy())
# probs_scratch = np.concatenate(probs_scratch_list, axis=0)

# probs_db_list = []
# with torch.no_grad():
#     for batch in test_loader_db:
#         input_ids = batch['input_ids'].to(device)
#         attention_mask = batch['attention_mask'].to(device)
#         outputs = model_db(input_ids=input_ids, attention_mask=attention_mask)
#         probs_db_list.append(torch.softmax(outputs.logits, dim=1).cpu().numpy())
# probs_db = np.concatenate(probs_db_list, axis=0)

# # Weighted Average Ensemble
# final_probs = 0.3 * probs_scratch + 0.7 * probs_db

# test_predictions = []
# choice_letters = ['A', 'B', 'C', 'D', 'E']
# for p in final_probs:
#     sorted_indices = np.argsort(p)[::-1][:3]
#     pred_str = " ".join([choice_letters[idx] for idx in sorted_indices])
#     test_predictions.append(pred_str)

# # Save submission CSV matching competition ID header format
# submission_df = pd.DataFrame({
#     'ID': test_df['id'],
#     'Prediction': test_predictions
# })
# submission_df.to_csv(OUTPUT_PATH, index=False)
# print(f"Success! Ensembled submission file saved to '{OUTPUT_PATH}'")

The PreTrained Model

In [ ]:
# import os
# import re
# import numpy as np
# import pandas as pd
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import Dataset, DataLoader
# from sklearn.model_selection import train_test_split

# # ----------------------------------------------------
# # 1. Hyperparameters & Configuration
# # ----------------------------------------------------
# TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
# TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
# OUTPUT_PATH = "submission.csv"

# # Seeding for reproducibility
# def set_seed(seed=42):
#     torch.manual_seed(seed)
#     torch.cuda.manual_seed_all(seed)
#     np.random.seed(seed)
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False

# set_seed(42)

# MAX_LEN = 256         # No truncation for long options
# BATCH_SIZE = 32
# EPOCHS = 12           # Smooth training over 12 epochs
# LEARNING_RATE = 8e-4  # Slightly lower learning rate for stable convergence
# EMBED_DIM = 256
# HIDDEN_DIM = 256
# NUM_LAYERS = 2
# DROPOUT_RATE = 0.4

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")

# PAD_TOKEN = "<PAD>"
# UNK_TOKEN = "<UNK>"
# SEP_TOKEN = "<SEP>"

# # ----------------------------------------------------
# # 2. Text Preprocessing & Vocabulary
# # ----------------------------------------------------
# def clean_text(text):
#     if not isinstance(text, str):
#         return ""
#     text = text.lower()
#     text = re.sub(r"[^\w\s]", "", text)  # Strip punctuation cleanly
#     text = re.sub(r"\s+", " ", text).strip()
#     return text

# class MCQVocabulary:
#     def __init__(self, max_vocab_size=20000):
#         self.max_vocab_size = max_vocab_size
#         self.word2idx = {PAD_TOKEN: 0, UNK_TOKEN: 1, SEP_TOKEN: 2}
#         self.idx2word = {0: PAD_TOKEN, 1: UNK_TOKEN, 2: SEP_TOKEN}
        
#     def fit(self, texts):
#         word_counts = {}
#         for text in texts:
#             cleaned = clean_text(text)
#             for word in cleaned.split():
#                 word_counts[word] = word_counts.get(word, 0) + 1
                
#         sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
#         for word, count in sorted_words[:self.max_vocab_size]:
#             if word not in self.word2idx:
#                 idx = len(self.word2idx)
#                 self.word2idx[word] = idx
#                 self.idx2word[idx] = word
                
#     def encode(self, text, max_len=128):
#         cleaned = clean_text(text)
#         tokens = cleaned.split()
#         encoded = [self.word2idx.get(w, self.word2idx[UNK_TOKEN]) for w in tokens]
        
#         if len(encoded) > max_len:
#             encoded = encoded[:max_len]
#         else:
#             encoded = encoded + [self.word2idx[PAD_TOKEN]] * (max_len - len(encoded))
#         return encoded

# # ----------------------------------------------------
# # 3. Dataset Pipeline
# # ----------------------------------------------------
# class ScratchMCQDataset(Dataset):
#     def __init__(self, df, vocab, max_len=128, is_train=True):
#         self.df = df.reset_index(drop=True)
#         self.vocab = vocab
#         self.max_len = max_len
#         self.is_train = is_train
#         self.label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
        
#     def __len__(self):
#         return len(self.df)
        
#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]
#         prompt = str(row['prompt'])
        
#         input_ids = []
#         for choice in ['A', 'B', 'C', 'D', 'E']:
#             choice_text = str(row[choice])
#             combined_text = f"{prompt} {SEP_TOKEN} {choice_text}"
#             encoded = self.vocab.encode(combined_text, max_len=self.max_len)
#             input_ids.append(encoded)
            
#         input_ids = torch.tensor(input_ids, dtype=torch.long)
        
#         if self.is_train and 'answer' in row:
#             label = torch.tensor(self.label_map[row['answer']], dtype=torch.long)
#             return input_ids, label
#         return input_ids

# # ----------------------------------------------------
# # 4. Neural Network Architecture
# # ----------------------------------------------------
# class SelfAttentionPooling(nn.Module):
#     def __init__(self, hidden_dim):
#         super().__init__()
#         self.attention = nn.Sequential(
#             nn.Linear(hidden_dim, hidden_dim // 2),
#             nn.Tanh(),
#             nn.Linear(hidden_dim // 2, 1)
#         )
        
#     def forward(self, rnn_outputs):
#         weights = self.attention(rnn_outputs)
#         weights = torch.softmax(weights, dim=1)
#         pooled = torch.sum(rnn_outputs * weights, dim=1)
#         return pooled

# class TunedBiGRUAttentionMCQModel(nn.Module):
#     def __init__(self, vocab_size, embed_dim=256, hidden_dim=256, num_layers=2, dropout=0.4):
#         super().__init__()
#         self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
#         self.gru = nn.GRU(
#             embed_dim, 
#             hidden_dim, 
#             batch_first=True, 
#             bidirectional=True, 
#             num_layers=num_layers,
#             dropout=dropout
#         )
#         self.attention = SelfAttentionPooling(hidden_dim * 2)
#         self.classifier = nn.Sequential(
#             nn.Linear(hidden_dim * 2, hidden_dim),
#             nn.LayerNorm(hidden_dim),
#             nn.ReLU(),
#             nn.Dropout(dropout),
#             nn.Linear(hidden_dim, 1)
#         )
        
#     def forward(self, input_ids):
#         batch_size, num_choices, seq_len = input_ids.shape
#         flat_input = input_ids.view(batch_size * num_choices, seq_len)
        
#         embedded = self.embedding(flat_input)
#         rnn_out, _ = self.gru(embedded)
        
#         pooled = self.attention(rnn_out)
#         logits = self.classifier(pooled)
        
#         return logits.view(batch_size, num_choices)

# # ----------------------------------------------------
# # 5. Training Pipeline
# # ----------------------------------------------------
# train_df = pd.read_csv(TRAIN_PATH).fillna("")
# test_df = pd.read_csv(TEST_PATH).fillna("")
# print(f"Dataset Loaded! Train shape: {train_df.shape}, Test shape: {test_df.shape}")

# # Fit Vocabulary ONLY on the train set (keeps test set out-of-vocabulary mapped to stable UNK)
# print("Fitting vocabulary...")
# corpus = train_df['prompt'].tolist()
# for col in ['A', 'B', 'C', 'D', 'E']:
#     corpus.extend(train_df[col].tolist())
# vocab = MCQVocabulary()
# vocab.fit(corpus)
# vocab_size = len(vocab.word2idx)
# print(f"Vocabulary Size: {vocab_size}")

# # Stratified Split (no deduplication for V7 compliance)
# train_data, val_data = train_test_split(train_df, test_size=0.1, random_state=42, stratify=train_df['answer'])

# train_dataset = ScratchMCQDataset(train_data, vocab, max_len=MAX_LEN, is_train=True)
# val_dataset = ScratchMCQDataset(val_data, vocab, max_len=MAX_LEN, is_train=True)
# test_dataset = ScratchMCQDataset(test_df, vocab, max_len=MAX_LEN, is_train=False)

# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
# test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# # Initialize Model
# model = TunedBiGRUAttentionMCQModel(
#     vocab_size=vocab_size, 
#     embed_dim=EMBED_DIM, 
#     hidden_dim=HIDDEN_DIM, 
#     num_layers=NUM_LAYERS,
#     dropout=DROPOUT_RATE
# ).to(device)

# criterion = nn.CrossEntropyLoss()
# optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
# scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# # Training loop
# for epoch in range(EPOCHS):
#     model.train()
#     train_loss = 0.0
#     correct = 0
#     total = 0
    
#     for batch_x, batch_y in train_loader:
#         batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
#         optimizer.zero_grad()
#         logits = model(batch_x)
#         loss = criterion(logits, batch_y)
#         loss.backward()
        
#         nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#         optimizer.step()
        
#         train_loss += loss.item()
#         preds = torch.argmax(logits, dim=1)
#         correct += (preds == batch_y).sum().item()
#         total += batch_y.size(0)
        
#     scheduler.step()
#     train_loss /= len(train_loader)
#     train_acc = correct / total
    
#     # Validation loop
#     model.eval()
#     val_loss = 0.0
#     val_correct = 0
#     val_total = 0
#     with torch.no_grad():
#         for batch_x, batch_y in val_loader:
#             batch_x, batch_y = batch_x.to(device), batch_y.to(device)
#             logits = model(batch_x)
#             loss = criterion(logits, batch_y)
            
#             val_loss += loss.item()
#             preds = torch.argmax(logits, dim=1)
#             val_correct += (preds == batch_y).sum().item()
#             val_total += batch_y.size(0)
            
#     val_loss /= len(val_loader)
#     val_acc = val_correct / val_total
    
#     current_lr = optimizer.param_groups[0]['lr']
#     print(f"Epoch {epoch+1}/{EPOCHS} | LR: {current_lr:.6f} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

# # ----------------------------------------------------
# # 6. Test Set Inference & Submission Formatting
# # ----------------------------------------------------
# print("Running inference on test set...")
# model.eval()
# test_predictions = []
# choice_letters = ['A', 'B', 'C', 'D', 'E']

# with torch.no_grad():
#     for batch_x in test_loader:
#         batch_x = batch_x.to(device)
#         logits = model(batch_x)
#         probs = torch.softmax(logits, dim=1).cpu().numpy()
        
#         for p in probs:
#             sorted_indices = np.argsort(p)[::-1][:3]
#             pred_str = " ".join([choice_letters[idx] for idx in sorted_indices])
#             test_predictions.append(pred_str)

# # Create submission DataFrame
# submission_df = pd.DataFrame({
#     'ID': test_df['id'],
#     'Prediction': test_predictions
# })

# # Save to submission CSV
# submission_df.to_csv(OUTPUT_PATH, index=False)
# print(f"Success! Submission file saved to '{OUTPUT_PATH}'")

Milestone 3

In [ ]:
# Install FAISS and Sentence Transformers in the Kaggle environment
!pip install -q faiss-cpu sentence-transformers

import os
import re
import numpy as np
import pandas as pd
import torch
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline

# ----------------------------------------------------
# 1. Configuration & Data Loading
# ----------------------------------------------------
TRAIN_PATH = "train.csv"

# Scan to locate train.csv dynamically (in case of folder structure variations)
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file == 'train.csv':
            TRAIN_PATH = os.path.join(root, file)

print(f"Loading dataset from: {TRAIN_PATH}\n")
train = pd.read_csv(TRAIN_PATH).fillna("")

# ----------------------------------------------------
# 2. Knowledge Base & FAISS Index Setup
# ----------------------------------------------------
print("Creating Knowledge Base (KB)...")
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading Sentence Transformer model and creating FAISS index...") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)
print("FAISS Index successfully initialized.\n")

# Helper to calculate average precision at 3
def calculate_ap3(predicted_letters, correct_letter):
    for rank, letter in enumerate(predicted_letters[:3]):
        if letter == correct_letter:
            return 1.0 / (rank + 1)
    return 0.0

# ----------------------------------------------------
# Q1. Zero-shot classifier (Softmax) on prompt 150
# ----------------------------------------------------
print("Running Task 1: Zero-shot Classification on Prompt 150...")
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0 if torch.cuda.is_available() else -1)

row_150 = train.iloc[150] 
prompt_150 = str(row_150['prompt']) 
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 
ans_150 = str(row_150[row_150['answer']])

res_q1 = zs(prompt_150, candidate_labels=labels_150, multi_label=False)
correct_idx_q1 = res_q1['labels'].index(ans_150)
q1_score = round(res_q1['scores'][correct_idx_q1], 3)
print(f"-> Q1 Score: {q1_score}\n")

# ----------------------------------------------------
# Q2. Query FAISS index for row index 150
# ----------------------------------------------------
print("Running Task 2: FAISS retrieval top k=10 rank...")
prompt_150_emb = model.encode([prompt_150], show_progress_bar=False) 
_, indices = index.search(prompt_150_emb, 10)
retrieved_indices = indices[0].tolist()

# Rank is 1-indexed
try:
    q2_rank = retrieved_indices.index(150) + 1
except ValueError:
    q2_rank = "Not in top 10"
print(f"-> Q2 Rank: {q2_rank}\n")

# ----------------------------------------------------
# Q3. Cross-Encoder reranking
# ----------------------------------------------------
print("Running Task 3: Cross-Encoder Reranking...")
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10 = [kb[i] for i in retrieved_indices]
pairs = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

# Rank documents based on scores
sorted_indices_ce = np.argsort(ce_scores)[::-1]
reranked_indices = [retrieved_indices[i] for i in sorted_indices_ce]

try:
    q3_rank = reranked_indices.index(150) + 1
except ValueError:
    q3_rank = "Not in top 10"
print(f"-> Q3 Rank: {q3_rank}\n")

# ----------------------------------------------------
# Q4. Token counts for prompt 42 with k=5 docs
# ----------------------------------------------------
print("Running Task 4: BERT Tokenizer Token Count on Prompt 42...")
row_42 = train.iloc[42] 
prompt_42 = str(row_42['prompt']) 

prompt_42_emb = model.encode([prompt_42], show_progress_bar=False) 
_, indices_42 = index.search(prompt_42_emb, 5)
retrieved_indices_42 = indices_42[0].tolist()

docs_5_42 = [kb[i] for i in retrieved_indices_42]
concatenated_docs_42 = " ".join(docs_5_42)
rag_string_42 = f"Context: {concatenated_docs_42} Question: {prompt_42}"

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
tokenized_42 = bert_tokenizer(rag_string_42, add_special_tokens=True)
q4_tokens = len(tokenized_42['input_ids'])
print(f"-> Q4 Token count: {q4_tokens}\n")

# ----------------------------------------------------
# Q5. RAG with correct true context for row index 150
# ----------------------------------------------------
print("Running Task 5: RAG zero-shot with true context...")
true_doc_150 = kb[150]
rag_string_150 = f"Context: {true_doc_150} Question: {prompt_150}"

res_q5 = zs(rag_string_150, candidate_labels=labels_150, multi_label=False)
correct_idx_q5 = res_q5['labels'].index(ans_150)
q5_score = round(res_q5['scores'][correct_idx_q5], 3)
print(f"-> Q5 Score: {q5_score}\n")

# ----------------------------------------------------
# Q6. Adversarial RAG (Context = KB index 999)
# ----------------------------------------------------
print("Running Task 6: Adversarial RAG context...")
bad_doc_999 = kb[999]
rag_string_999 = f"Context: {bad_doc_999} Question: {prompt_150}"

res_q6 = zs(rag_string_999, candidate_labels=labels_150, multi_label=False)
correct_idx_q6 = res_q6['labels'].index(ans_150)
q6_score = round(res_q6['scores'][correct_idx_q6], 3)
print(f"-> Q6 Score: {q6_score}\n")

# ----------------------------------------------------
# Q7. Hit Rate percentage for first 100 rows
# ----------------------------------------------------
print("Running Task 7: Hit Rate percentage over first 100 rows...")
hits = 0
for i in range(100):
    row_i = train.iloc[i]
    prompt_i = str(row_i['prompt'])
    correct_option_i = str(row_i[row_i['answer']])
    
    prompt_i_emb = model.encode([prompt_i], show_progress_bar=False) 
    _, indices_i = index.search(prompt_i_emb, 5)
    retrieved_indices_i = indices_i[0].tolist()
    docs_5_i = [kb[idx] for idx in retrieved_indices_i]
    
    # Hit if the exact correct choice string is retrieved in the top 5 documents
    if any(correct_option_i == doc for doc in docs_5_i):
        hits += 1

hit_rate = (hits / 100.0) * 100.0
print(f"-> Q7 Hit Rate: {round(hit_rate, 1)}%\n")

# ----------------------------------------------------
# Q8. RAG Pipeline MAP@3 over first 20 rows
# ----------------------------------------------------
print("Running Task 8: MAP@3 of two-stage RAG Pipeline over first 20 rows...")
ap3_scores = []
for i in range(20):
    row_i = train.iloc[i]
    prompt_i = str(row_i['prompt'])
    correct_letter_i = row_i['answer']
    
    options_map = {
        str(row_i['A']): 'A',
        str(row_i['B']): 'B',
        str(row_i['C']): 'C',
        str(row_i['D']): 'D',
        str(row_i['E']): 'E'
    }
    candidate_labels_i = list(options_map.keys())
    
    # 1. Retrieve top 5 from FAISS
    prompt_i_emb = model.encode([prompt_i], show_progress_bar=False) 
    _, indices_i = index.search(prompt_i_emb, 5)
    retrieved_indices_i = indices_i[0].tolist()
    docs_5_i = [kb[idx] for idx in retrieved_indices_i]
    
    # 2. Rerank with Cross-Encoder
    pairs_i = [[prompt_i, doc] for doc in docs_5_i]
    ce_scores_i = cross_encoder.predict(pairs_i)
    best_doc_idx = np.argmax(ce_scores_i)
    best_doc_i = docs_5_i[best_doc_idx]
    
    # 3. Augment
    rag_string_i = f"Context: {best_doc_i} Question: {prompt_i}"
    
    # 4. Predict
    res_i = zs(rag_string_i, candidate_labels=candidate_labels_i, multi_label=False)
    
    # Sort predictions
    ranked_labels = res_i['labels']
    ranked_letters = [options_map[lbl] for lbl in ranked_labels]
    
    # AP@3 score
    ap3 = calculate_ap3(ranked_letters, correct_letter_i)
    ap3_scores.append(ap3)

mean_ap3 = np.mean(ap3_scores)
print(f"-> Q8 Final average MAP@3: {round(mean_ap3, 3)}\n")

# ----------------------------------------------------
# Final Summary Output
# ----------------------------------------------------
print("====================================")
print("       MILESTONE 3 ANSWERS          ")
print("====================================")
print(f"Q1: {q1_score}")
print(f"Q2: {q2_rank}")
print(f"Q3: {q3_rank}")
print(f"Q4: {q4_tokens}")
print(f"Q5: {q5_score}")
print(f"Q6: {q6_score}")
print(f"Q7: {round(hit_rate, 1)}")
print(f"Q8: {round(mean_ap3, 3)}")
print("====================================")

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import random

# ----------------------------------------------------
# 1. Configuration & Seeding
# ----------------------------------------------------
TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
OUTPUT_PATH = "submission.csv"

# Dynamically locate paths if mounted differently
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file == 'train.csv':
            TRAIN_PATH = os.path.join(root, file)
        elif file == 'test.csv':
            TEST_PATH = os.path.join(root, file)

print(f"Loaded Train Path: {TRAIN_PATH}")
print(f"Loaded Test Path: {TEST_PATH}")

MAX_LEN = 256
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001
EMBED_DIM = 256
HIDDEN_DIM = 256

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
SEP_TOKEN = "<SEP>"

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ----------------------------------------------------
# 2. Text Preprocessing (Strips Punctuation Cleanly)
# ----------------------------------------------------
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)  # Strips all punctuation to unify tokens
    text = re.sub(r"\s+", " ", text).strip()
    return text

class MCQVocabulary:
    def __init__(self, max_vocab_size=20000):
        self.max_vocab_size = max_vocab_size
        self.word2idx = {PAD_TOKEN: 0, UNK_TOKEN: 1, SEP_TOKEN: 2}
        self.idx2word = {0: PAD_TOKEN, 1: UNK_TOKEN, 2: SEP_TOKEN}
       
    def fit(self, texts):
        word_counts = {}
        for text in texts:
            cleaned = clean_text(text)
            for word in cleaned.split():
                word_counts[word] = word_counts.get(word, 0) + 1
               
        sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
        for word, count in sorted_words[:self.max_vocab_size]:
            if word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word
               
    def encode(self, text, max_len=MAX_LEN):
        cleaned = clean_text(text)
        tokens = cleaned.split()
        encoded = [self.word2idx.get(w, self.word2idx[UNK_TOKEN]) for w in tokens]
       
        if len(encoded) > max_len:
            encoded = encoded[:max_len]
        else:
            encoded = encoded + [self.word2idx[PAD_TOKEN]] * (max_len - len(encoded))
        return encoded

# ----------------------------------------------------
# 3. Dataset Class
# ----------------------------------------------------
class ScratchMCQDataset(Dataset):
    def __init__(self, df, vocab, max_len=MAX_LEN, is_train=True):
        self.df = df.reset_index(drop=True)
        self.vocab = vocab
        self.max_len = max_len
        self.is_train = is_train
        self.label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
       
    def __len__(self):
        return len(self.df)
       
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt'])
       
        input_ids = []
        for choice in ['A', 'B', 'C', 'D', 'E']:
            choice_text = str(row[choice])
            combined_text = f"{prompt} {SEP_TOKEN} {choice_text}"
            encoded = self.vocab.encode(combined_text, max_len=self.max_len)
            input_ids.append(encoded)
           
        input_ids = torch.tensor(input_ids, dtype=torch.long)
       
        if self.is_train and 'answer' in row:
            label = torch.tensor(self.label_map[row['answer']], dtype=torch.long)
            return input_ids, label
        return input_ids

# ----------------------------------------------------
# 4. Neural Network (BiLSTM + Attention)
# ----------------------------------------------------
class SelfAttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )
       
    def forward(self, rnn_outputs):
        weights = self.attention(rnn_outputs)
        weights = torch.softmax(weights, dim=1)
        pooled = torch.sum(rnn_outputs * weights, dim=1)
        return pooled

class ImprovedBiLSTMAttentionMCQModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=256, dropout=0.4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, batch_first=True,
            bidirectional=True, num_layers=2, dropout=dropout
        )
        self.attention = SelfAttentionPooling(hidden_dim * 2)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )
       
    def forward(self, input_ids):
        batch_size, num_choices, seq_len = input_ids.shape
        flat_input = input_ids.view(batch_size * num_choices, seq_len)
       
        embedded = self.embedding(flat_input)
        lstm_out, _ = self.lstm(embedded)
       
        pooled = self.attention(lstm_out)
        logits = self.classifier(pooled)
       
        return logits.view(batch_size, num_choices)

# ----------------------------------------------------
# 5. Core Execution (With Global Failsafe)
# ----------------------------------------------------
try:
    # Load Data
    train_raw = pd.read_csv(TRAIN_PATH).fillna("")
    test_df = pd.read_csv(TEST_PATH).fillna("")
    print(f"Loaded | Train: {train_raw.shape}, Test: {test_df.shape}")

    # Fit Vocabulary ONLY on the training set to prevent untrained test embeddings
    print("Fitting Vocabulary on Train set...")
    corpus = train_raw['prompt'].tolist()
    for col in ['A', 'B', 'C', 'D', 'E']:
        corpus.extend(train_raw[col].tolist())

    vocab = MCQVocabulary()
    vocab.fit(corpus)
    vocab_size = len(vocab.word2idx)
    print(f"Vocabulary Size: {vocab_size}")

    # Train / Val Split (no deduplication for maximum training signal)
    train_data, val_data = train_test_split(train_raw, test_size=0.1, random_state=42, stratify=train_raw['answer'])

    train_dataset = ScratchMCQDataset(train_data, vocab, is_train=True)
    val_dataset = ScratchMCQDataset(val_data, vocab, is_train=True)
    test_dataset = ScratchMCQDataset(test_df, vocab, is_train=False)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # Training Loop
    def train_one_model(seed):
        set_seed(seed)
        print(f"\nTraining model with seed {seed}")
       
        model = ImprovedBiLSTMAttentionMCQModel(vocab_size, EMBED_DIM, HIDDEN_DIM).to(device)
        criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
        optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
       
        for epoch in range(EPOCHS):
            model.train()
            train_loss, correct, total = 0.0, 0, 0
           
            for batch_x, batch_y in train_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
               
                optimizer.zero_grad()
                logits = model(batch_x)
                loss = criterion(logits, batch_y)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
               
                train_loss += loss.item()
                preds = torch.argmax(logits, dim=1)
                correct += (preds == batch_y).sum().item()
                total += batch_y.size(0)
           
            scheduler.step()
           
            # Validation
            model.eval()
            val_correct, val_total = 0, 0
            with torch.no_grad():
                for batch_x, batch_y in val_loader:
                    batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                    logits = model(batch_x)
                    preds = torch.argmax(logits, dim=1)
                    val_correct += (preds == batch_y).sum().item()
                    val_total += batch_y.size(0)
           
            print(f"Epoch {epoch+1}/{EPOCHS} | Train Acc: {correct/total:.4f} | Val Acc: {val_correct/val_total:.4f}")
       
        return model

    # Train three models for ensembling
    model1 = train_one_model(42)
    model2 = train_one_model(123)
    model3 = train_one_model(2026)

    # Ensemble Inference
    print("\nRunning ensemble inference (Seeds 42 + 123 + 2026)...")
    model1.eval()
    model2.eval()
    model3.eval()

    test_predictions = []
    choice_letters = ['A', 'B', 'C', 'D', 'E']

    with torch.no_grad():
        for batch_x in test_loader:
            batch_x = batch_x.to(device)
           
            logits1 = model1(batch_x)
            logits2 = model2(batch_x)
            logits3 = model3(batch_x)
           
            # Softmax probability average
            probs = (torch.softmax(logits1, dim=1) + 
                     torch.softmax(logits2, dim=1) + 
                     torch.softmax(logits3, dim=1)) / 3
            probs = probs.cpu().numpy()
           
            for p in probs:
                sorted_idx = np.argsort(p)[::-1][:3]
                pred_str = " ".join([choice_letters[i] for i in sorted_idx])
                test_predictions.append(pred_str)

    # Write submission
    submission_df = pd.DataFrame({
        'ID': test_df['id'],
        'Prediction': test_predictions
    })
    submission_df.to_csv(OUTPUT_PATH, index=False)
    print(f"Submission saved → {OUTPUT_PATH}")

except Exception as e:
    print(f"\n[CRITICAL ERROR] Pipeline crashed: {e}")
    print("Falling back to safe predictions baseline to guarantee output...")
    try:
        test_df = pd.read_csv(TEST_PATH).fillna("")
        fallback_preds = ["B C A"] * len(test_df)
        submission_df = pd.DataFrame({
            'ID': test_df['id'],
            'Prediction': fallback_preds
        })
        submission_df.to_csv(OUTPUT_PATH, index=False)
        print("Success! Failsafe 'submission.csv' generated.")
    except Exception as fallback_err:
        print(f"Fallback generation failed: {fallback_err}")

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import random

# Tabular Boosters
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# ----------------------------------------------------
# 1. Configuration & Seeding
# ----------------------------------------------------
TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
OUTPUT_PATH = "submission.csv"

for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file == 'train.csv':
            TRAIN_PATH = os.path.join(root, file)
        elif file == 'test.csv':
            TEST_PATH = os.path.join(root, file)

print(f"Using Train Path: {TRAIN_PATH}")
print(f"Using Test Path: {TEST_PATH}")

MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 1e-3

# Strongly reduced capacity to prevent memorization
EMBED_DIM = 64
HIDDEN_DIM = 64
DROPOUT_RATE = 0.5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
SEP_TOKEN = "<SEP>"

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ----------------------------------------------------
# 2. Preprocessing & Feature Extraction
# ----------------------------------------------------
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)  # Strips punctuation
    text = re.sub(r"\s+", " ", text).strip()
    return text

def extract_tabular_features(df, tfidf_vectorizer=None, is_train=True):
    prompts = [clean_text(p) for p in df['prompt'].tolist()]
    
    if is_train:
        tfidf_vectorizer = TfidfVectorizer(stop_words='english')
        tfidf_vectorizer.fit(prompts)
        
    features = []
    for idx, row in df.iterrows():
        p_vec = tfidf_vectorizer.transform([prompts[idx]])
        p_words = set(prompts[idx].split())
        
        row_feats = []
        for choice in ['A', 'B', 'C', 'D', 'E']:
            choice_text = clean_text(str(row[choice]))
            c_vec = tfidf_vectorizer.transform([choice_text])
            c_words = set(choice_text.split())
            
            # Feature 1: TF-IDF similarity
            sim = cosine_similarity(p_vec, c_vec)[0][0]
            
            # Feature 2: Word overlap ratio
            overlap_c = len(p_words.intersection(c_words))
            overlap = overlap_c / (len(c_words) + 1e-8)
            
            # Feature 3: Jaccard similarity (Intersection over Union)
            union_words = p_words.union(c_words)
            jaccard = len(p_words.intersection(c_words)) / (len(union_words) + 1e-8)
            
            # Feature 4: Choice character length
            c_len = len(str(row[choice]))
            
            # Feature 5: Choice length ratio
            len_ratio = c_len / (len(str(row['prompt'])) + 1e-8)
            
            row_feats.extend([sim, overlap, jaccard, c_len, len_ratio])
            
        features.append(row_feats)
        
    return np.array(features), tfidf_vectorizer

# ----------------------------------------------------
# 3. Vocabulary & PyTorch Dataset
# ----------------------------------------------------
class MCQVocabulary:
    def __init__(self, max_vocab_size=15000):
        self.max_vocab_size = max_vocab_size
        self.word2idx = {PAD_TOKEN: 0, UNK_TOKEN: 1, SEP_TOKEN: 2}
        self.idx2word = {0: PAD_TOKEN, 1: UNK_TOKEN, 2: SEP_TOKEN}
       
    def fit(self, texts):
        word_counts = {}
        for text in texts:
            cleaned = clean_text(text)
            for word in cleaned.split():
                word_counts[word] = word_counts.get(word, 0) + 1
               
        sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
        for word, count in sorted_words[:self.max_vocab_size]:
            if word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word
               
    def encode(self, text, max_len=MAX_LEN):
        cleaned = clean_text(text)
        tokens = cleaned.split()
        encoded = [self.word2idx.get(w, self.word2idx[UNK_TOKEN]) for w in tokens]
       
        if len(encoded) > max_len:
            encoded = encoded[:max_len]
        else:
            encoded = encoded + [self.word2idx[PAD_TOKEN]] * (max_len - len(encoded))
        return encoded

class ScratchMCQDataset(Dataset):
    def __init__(self, df, vocab, max_len=MAX_LEN, is_train=True):
        self.df = df.reset_index(drop=True)
        self.vocab = vocab
        self.max_len = max_len
        self.is_train = is_train
        self.label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
       
    def __len__(self):
        return len(self.df)
       
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt'])
       
        input_ids = []
        for choice in ['A', 'B', 'C', 'D', 'E']:
            choice_text = str(row[choice])
            combined_text = f"{prompt} {SEP_TOKEN} {choice_text}"
            encoded = self.vocab.encode(combined_text, max_len=self.max_len)
            input_ids.append(encoded)
           
        input_ids = torch.tensor(input_ids, dtype=torch.long)
       
        if self.is_train and 'answer' in row:
            label = torch.tensor(self.label_map[row['answer']], dtype=torch.long)
            return input_ids, label
        return input_ids

# ----------------------------------------------------
# 4. Neural Network (BiGRU - 1 Layer)
# ----------------------------------------------------
class SelfAttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )
       
    def forward(self, rnn_outputs):
        weights = self.attention(rnn_outputs)
        weights = torch.softmax(weights, dim=1)
        pooled = torch.sum(rnn_outputs * weights, dim=1)
        return pooled

class BiGRUAttentionMCQModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=64, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(
            embed_dim, hidden_dim, batch_first=True, bidirectional=True
        )
        self.attention = SelfAttentionPooling(hidden_dim * 2)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )
       
    def forward(self, input_ids):
        batch_size, num_choices, seq_len = input_ids.shape
        flat_input = input_ids.view(batch_size * num_choices, seq_len)
       
        embedded = self.embedding(flat_input)
        rnn_out, _ = self.gru(embedded)
       
        pooled = self.attention(rnn_out)
        logits = self.classifier(pooled)
       
        return logits.view(batch_size, num_choices)

# ----------------------------------------------------
# 5. Core Execution (With Global Failsafe)
# ----------------------------------------------------
try:
    # Load Data
    train_df = pd.read_csv(TRAIN_PATH).fillna("")
    test_df = pd.read_csv(TEST_PATH).fillna("")
    print(f"Loaded | Train: {train_df.shape}, Test: {test_df.shape}")

    # --- PART A: Deep Tabular Boosting Models ---
    print("\n--- Preparing Tabular Features ---")
    X_train_tab, tfidf_vec = extract_tabular_features(train_df, is_train=True)
    X_test_tab, _ = extract_tabular_features(test_df, tfidf_vectorizer=tfidf_vec, is_train=False)
    
    label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
    y_train = np.array([label_map[ans] for ans in train_df['answer']])
    
    # Validation split for early stopping evaluation
    X_tr, X_va, y_tr, y_va = train_test_split(
        X_train_tab, y_train, test_size=0.1, random_state=42, stratify=y_train
    )
    
    # Train Deep XGBoost Classifier
    print("\nTraining Deep XGBoost (max_depth=19, n_estimators=15k, early_stopping_rounds=150)...")
    xgb = XGBClassifier(
        n_estimators=15000,
        max_depth=19,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.8,
        early_stopping_rounds=150,
        verbosity=1,
        random_state=42
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=250)
    probs_xgb = xgb.predict_proba(X_test_tab)
    print("XGBoost training complete.")
    
    # Train Deep CatBoost Classifier (Iterations to 3000, early stopping to 150)
    print("\nTraining Deep CatBoost (depth=10, iterations=3k, early_stopping_rounds=150)...")
    cat = CatBoostClassifier(
        iterations=3000,
        depth=10,
        learning_rate=0.01,
        early_stopping_rounds=150,
        random_seed=42,
        verbose=300  # prints every 300 iterations
    )
    cat.fit(X_tr, y_tr, eval_set=(X_va, y_va))
    probs_cat = cat.predict_proba(X_test_tab)
    print("CatBoost training complete.")

    # --- PART B: Neural Networks (5 Seeds) ---
    print("\n--- Training 5 Neural Networks ---")
    corpus = train_df['prompt'].tolist()
    for col in ['A', 'B', 'C', 'D', 'E']:
        corpus.extend(train_df[col].tolist())

    vocab = MCQVocabulary()
    vocab.fit(corpus)
    vocab_size = len(vocab.word2idx)
    print(f"Vocabulary Size: {vocab_size}")

    train_dataset = ScratchMCQDataset(train_df, vocab, is_train=True)
    test_dataset = ScratchMCQDataset(test_df, vocab, is_train=False)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    def train_one_nn(seed):
        set_seed(seed)
        print(f"Training NN Model with seed {seed}...")
       
        model = BiGRUAttentionMCQModel(vocab_size, EMBED_DIM, HIDDEN_DIM, dropout=DROPOUT_RATE).to(device)
        criterion = nn.CrossEntropyLoss()
        # High weight decay (1e-2) in AdamW for aggressive regularizing
        optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)
       
        for epoch in range(EPOCHS):
            model.train()
            train_loss, correct, total = 0.0, 0, 0
            for batch_x, batch_y in train_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                optimizer.zero_grad()
                logits = model(batch_x)
                loss = criterion(logits, batch_y)
                loss.backward()
                optimizer.step()
               
                train_loss += loss.item()
                preds = torch.argmax(logits, dim=1)
                correct += (preds == batch_y).sum().item()
                total += batch_y.size(0)
            
            if (epoch+1) == EPOCHS:
                print(f"-> Finished Seed {seed} | Final Train Acc: {correct/total:.4f}")
       
        return model

    # Train NNs
    nn_seeds = [42, 123, 2026, 888, 999]
    nn_models = [train_one_nn(s) for s in nn_seeds]

    # Inference NNs
    probs_nns = []
    with torch.no_grad():
        for batch_x in test_loader:
            batch_x = batch_x.to(device)
            batch_probs = []
            for model in nn_models:
                model.eval()
                logits = model(batch_x)
                batch_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
            
            # Average probabilities of 5 seeds
            avg_batch_probs = np.mean(batch_probs, axis=0)
            probs_nns.append(avg_batch_probs)
            
    probs_nn = np.concatenate(probs_nns, axis=0)

    # --- PART C: Blending/Ensembling ---
    print("\nBlending predictions...")
    # 60% NN ensemble + 20% XGBoost + 20% CatBoost
    final_probs = 0.60 * probs_nn + 0.20 * probs_xgb + 0.20 * probs_cat

    test_predictions = []
    choice_letters = ['A', 'B', 'C', 'D', 'E']
    for p in final_probs:
        sorted_idx = np.argsort(p)[::-1][:3]
        pred_str = " ".join([choice_letters[i] for i in sorted_idx])
        test_predictions.append(pred_str)

    # Write submission file
    submission_df = pd.DataFrame({
        'ID': test_df['id'],
        'Prediction': test_predictions
    })
    submission_df.to_csv(OUTPUT_PATH, index=False)
    print(f"Success! Ensembled submission saved → {OUTPUT_PATH}")

except Exception as e:
    print(f"\n[CRITICAL ERROR] Pipeline crashed: {e}")
    print("Falling back to safe predictions baseline to guarantee output...")
    try:
        test_df = pd.read_csv(TEST_PATH).fillna("")
        fallback_preds = ["B C A"] * len(test_df)
        submission_df = pd.DataFrame({
            'ID': test_df['id'],
            'Prediction': fallback_preds
        })
        submission_df.to_csv(OUTPUT_PATH, index=False)
        print("Success! Failsafe 'submission.csv' generated.")
    except Exception as fallback_err:
        print(f"Fallback generation failed: {fallback_err}")